# 6.1 Model monitor (run in SageMaker Studio)

`ModelQualityMonitor` (multiclass) baselined on **test** predictions vs labels, then a one-off analyzer job on the **production** predictions vs `true_activity_id`, compared with `xgboost_eval.json`. Batch only: no endpoint or schedule.

**Caveat:** test and production windows come from subjects seen in training (chronological split, spec B), so these quality numbers are optimistic.

In [ ]:
# SageMaker Python SDK v3 (default on current Studio images) removed the v2 modules this notebook uses.
# Pin v2; on first run this installs it and restarts the kernel - then run the notebook again from the top.
from importlib.metadata import version, PackageNotFoundError
try:
    _sm_ver = version("sagemaker")
except PackageNotFoundError:
    _sm_ver = "0"
if int(_sm_ver.split(".")[0]) != 2:
    %pip uninstall -q -y sagemaker sagemaker-core sagemaker-train sagemaker-serve sagemaker-mlops
    %pip install -q "sagemaker>=2.200,<3"
    print(f"Replaced sagemaker {_sm_ver} with v2 - restarting kernel; re-run the notebook from the top.")
    import IPython
    IPython.Application.instance().kernel.do_shutdown(restart=True)
else:
    print("sagemaker", _sm_ver)

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import io, json, time
import boto3, numpy as np, pandas as pd
from botocore.exceptions import ClientError
from src.pamap2.config import load_config, s3_uri
from src.pamap2.schema import ACTIVITY_NAMES
from src.pamap2 import evaluation as E
from src.pamap2 import monitoring as M
cfg = load_config()
session = boto3.Session(region_name=cfg["region"])
print("Caller:", session.client("sts").get_caller_identity()["Arn"])
s3 = session.client("s3")
cw = session.client("cloudwatch")
sm = session.client("sagemaker")
SPLITS = s3_uri(cfg, "splits")
REPORTS = s3_uri(cfg, "models", "reports/")
MON = lambda key: s3_uri(cfg, "monitoring", key)
NS = cfg["monitoring_namespace"]

def split_key(uri):
    b, k = uri[len("s3://"):].split("/", 1)
    return b, k
def read_bytes(uri):
    b, k = split_key(uri)
    return s3.get_object(Bucket=b, Key=k)["Body"].read()
def read_bytes_or_none(uri):
    try:
        return read_bytes(uri)
    except ClientError as e:
        if e.response["Error"]["Code"] in ("NoSuchKey", "404"):
            return None
        raise
def read_parquet(name):
    return pd.read_parquet(io.BytesIO(read_bytes(SPLITS + name)))
def read_json(uri):
    return json.loads(read_bytes(uri))
def write_json(uri, obj):
    b, k = split_key(uri)
    s3.put_object(Bucket=b, Key=k, Body=json.dumps(obj, indent=2).encode())
    print("Wrote", uri)
def put_text(uri, text):
    b, k = split_key(uri)
    s3.put_object(Bucket=b, Key=k, Body=text.encode())
    return uri
def exists(uri):
    b, k = split_key(uri)
    try:
        s3.head_object(Bucket=b, Key=k)
        return True
    except ClientError:
        return False
def latest_subprefix(uri_prefix):
    """Newest child 'folder' (timestamped run folders sort lexicographically)."""
    b, k = split_key(uri_prefix)
    r = s3.list_objects_v2(Bucket=b, Prefix=k, Delimiter="/")
    subs = sorted(p["Prefix"] for p in r.get("CommonPrefixes", []))
    return f"s3://{b}/{subs[-1]}" if subs else None
def publish(metrics):
    """metrics: {name: value} -> custom namespace (put_metric_data)."""
    cw.put_metric_data(Namespace=NS, MetricData=[{"MetricName": n, "Value": float(v), "Unit": "None"} for n, v in metrics.items()])
    print("Published", metrics)

def safe_publish(metrics):
    try:
        publish(metrics)
    except Exception as e:   # never mask the original error
        print("WARNING: publish failed:", e)
def report_failure(job_name):
    try:
        print("FailureReason:", sm.describe_processing_job(ProcessingJobName=job_name).get("FailureReason"))
    except Exception as e:
        print("Could not describe job:", e)

FEATURES = read_json(SPLITS + "feature_columns.json")
LABEL_MAP = E.load_label_map(read_json(SPLITS + "label_map.json"))   # class index -> activity_id
K = len(LABEL_MAP)
ACTIVITY_IDS = list(LABEL_MAP.values())
print(K, "classes;", len(FEATURES), "features")


In [ ]:
import sagemaker
from sagemaker.processing import Processor, ProcessingInput, ProcessingOutput
sm_session = sagemaker.Session(boto_session=session)
ANALYZER = sagemaker.image_uris.retrieve("model-monitor", cfg["region"])

def run_analyzer(name, env, inputs, output_uri):
    """One-off Model Monitor analyzer job (ml.m5.large x1). inputs: [(s3_uri, container_dest, input_name)]."""
    proc = Processor(role=cfg["role_arn"], image_uri=ANALYZER, instance_count=1, instance_type="ml.m5.large",
                     volume_size_in_gb=20, max_runtime_in_seconds=3600, env=env, sagemaker_session=sm_session)
    proc.run(inputs=[ProcessingInput(source=s, destination=d, input_name=n) for s, d, n in inputs],
             outputs=[ProcessingOutput(source="/opt/ml/processing/output", destination=output_uri, output_name="result")],
             job_name=name, wait=True, logs=False)
    return output_uri
from sagemaker.model_monitor import ModelQualityMonitor
from sagemaker.model_monitor.dataset_format import DatasetFormat


## Baseline dataset: test predictions (from 4.2's batch-transform output) vs test labels

In [ ]:
test = read_parquet("test.parquet").reset_index(drop=True)
probs = E.parse_probs(read_bytes(s3_uri(cfg, "models", "transform-output/test/features.csv.out")), K)
assert len(probs) == len(test), (len(probs), len(test))
cls = E.probs_to_labels(probs, K)
test_preds = pd.DataFrame({"predicted_activity_id": [LABEL_MAP[int(c)] for c in cls], "confidence": probs.max(axis=1),
                           "true_activity_id": test["activity_id"]})
base_df, dropped = M.to_model_quality_frame(test_preds)
print("Baseline rows:", len(base_df), "| dropped NaN labels:", dropped)
base_uri = put_text(MON("model-quality/input/baseline.csv"), base_df.to_csv(index=False))
base_metrics = E.classification_metrics(base_df["label"], base_df["prediction"], ACTIVITY_IDS)
eval_report = read_json(REPORTS + "xgboost_eval.json")
print(f"Local test macro-F1={base_metrics['macro_f1']:.4f} | xgboost_eval.json macro-F1={eval_report['macro_f1']:.4f}")
if abs(base_metrics["macro_f1"] - eval_report["macro_f1"]) > 1e-3:
    print("WARNING: baseline macro-F1 differs from xgboost_eval.json by > 1e-3 - transform output may be stale")


## Baseline job (billable: one ml.m5.large processing job)

In [ ]:
BASELINE = MON("model-quality/baseline/")
mq = ModelQualityMonitor(role=cfg["role_arn"], instance_count=1, instance_type="ml.m5.large", volume_size_in_gb=20,
                         max_runtime_in_seconds=3600, sagemaker_session=sm_session)
mq.suggest_baseline(baseline_dataset=base_uri, dataset_format=DatasetFormat.csv(header=True), output_s3_uri=BASELINE,
                    problem_type="MulticlassClassification", inference_attribute="prediction",
                    probability_attribute="probability", ground_truth_attribute="label", wait=True, logs=False)
for f in ("statistics.json", "constraints.json"):
    print(BASELINE + f, "exists:", exists(BASELINE + f))


## Production predictions vs true_activity_id

In [ ]:
pb, pk = split_key(s3_uri(cfg, "models", "predictions/production/"))
objs = [o for p in s3.get_paginator("list_objects_v2").paginate(Bucket=pb, Prefix=pk)
        for o in p.get("Contents", []) if o["Key"].endswith(".parquet")]
if not objs:
    raise RuntimeError("no production prediction parquet found - run 5.0 first")
newest = max(objs, key=lambda o: o["LastModified"])["Key"]
stem = newest[:-len(".parquet")]
parts = [o for o in objs if o["Key"] == newest or o["Key"].startswith(stem + "/")]   # all parts of the latest 5.0 output
print("Using", [o["Key"] for o in parts])
preds = pd.concat([pd.read_parquet(io.BytesIO(s3.get_object(Bucket=pb, Key=o["Key"])["Body"].read())) for o in parts],
                  ignore_index=True)
n_prod = len(read_parquet("production.parquet"))   # split file is already restricted to train classes (= 5.0 input)
assert len(preds) == n_prod, f"predictions rows {len(preds)} != production split rows {n_prod}"
prod_df, dropped = M.to_model_quality_frame(preds)
if prod_df.empty:
    raise RuntimeError("no labelled production rows (true_activity_id all NaN)")
print("Production rows:", len(prod_df), "| dropped rows with NaN label/prediction/probability:", dropped)
prod_uri = put_text(MON("model-quality/input/production.csv"), prod_df.to_csv(index=False))
prod_metrics = E.classification_metrics(prod_df["label"], prod_df["prediction"], ACTIVITY_IDS)


## One-off monitoring job (billable: one ml.m5.large processing job)

In [ ]:
stamp = time.strftime("%Y%m%d-%H%M%S")
RUN = MON(f"model-quality/runs/{stamp}/")
env = {"analysis_type": "MODEL_QUALITY", "problem_type": "MulticlassClassification",
       "dataset_format": json.dumps(DatasetFormat.csv(header=True)), "dataset_source": "/opt/ml/processing/input/data",
       "inference_attribute": "prediction", "probability_attribute": "probability", "ground_truth_attribute": "label",
       "output_path": "/opt/ml/processing/output", "publish_cloudwatch_metrics": "Disabled",
       "baseline_constraints": "/opt/ml/processing/baseline/constraints/constraints.json",
       "baseline_statistics": "/opt/ml/processing/baseline/stats/statistics.json"}
job_name = f"{cfg['prefix']}-model-monitor-{stamp}"
try:
    run_analyzer(job_name, env, [
        (prod_uri, "/opt/ml/processing/input/data", "data"),
        (BASELINE + "constraints.json", "/opt/ml/processing/baseline/constraints", "constraints"),
        (BASELINE + "statistics.json", "/opt/ml/processing/baseline/stats", "statistics")], RUN)
except Exception:
    report_failure(job_name)
    safe_publish({"ModelJobFailed": 1})
    raise
safe_publish({"ModelJobFailed": 0})


## Violations and quality drift vs xgboost_eval.json

In [ ]:
assert exists(RUN + "statistics.json"), "no statistics.json in run output - job incomplete; absent violations would not mean zero"
v = M.parse_violations(read_bytes_or_none(RUN + "constraint_violations.json"))
print("Violations:", v["count"], "->", v["status"])
display(pd.DataFrame(v["violations"], columns=["feature", "check_type", "description"]))
drift = M.quality_drift(eval_report["macro_f1"], prod_metrics["macro_f1"], cfg["monitoring_drift_threshold"])
print(drift)
display(pd.DataFrame([{"metric": m, "baseline_eval": eval_report[m], "production": prod_metrics[m]} for m in ("macro_f1", "accuracy")]))
print("NOTE: seen-subject production windows make these numbers optimistic (spec B).")
write_json(RUN + "violations_summary.json", {**v, "drift": drift})
publish({"ModelMacroF1": prod_metrics["macro_f1"], "ModelAccuracy": prod_metrics["accuracy"],
         "ModelViolationCount": v["count"], "ModelDegraded": int(drift["degraded"])})


## Cleanup

In [ ]:
# Cleanup: nothing here creates schedules or endpoints; list them, delete only temporary pamap2-har-mon* models.
for page in sm.get_paginator("list_monitoring_schedules").paginate(NameContains=cfg["prefix"]):
    for x in page["MonitoringScheduleSummaries"]:
        print("WARNING: monitoring schedule exists (not deleted):", x["MonitoringScheduleName"])
for page in sm.get_paginator("list_models").paginate(NameContains=f"{cfg['prefix']}-mon"):
    for m in page["Models"]:
        if m["ModelName"].startswith(f"{cfg['prefix']}-mon"):
            print("Deleting temporary model", m["ModelName"])
            sm.delete_model(ModelName=m["ModelName"])
for kind, pager, key, field in (("processing", "list_processing_jobs", "ProcessingJobSummaries", "ProcessingJobName"),
                                ("transform", "list_transform_jobs", "TransformJobSummaries", "TransformJobName")):
    for page in sm.get_paginator(pager).paginate(StatusEquals="InProgress", NameContains=cfg["prefix"]):
        for x in page[key]:
            print("WARNING: still running", kind, "job:", x[field])
print("Endpoints:", [e["EndpointName"] for p in sm.get_paginator("list_endpoints").paginate() for e in p["Endpoints"]])
